In [1]:
import requests
import json
import pandas as pd

print("Requests library imported successfully!")

Requests library imported successfully!


In [2]:
#define api endpoint and parameters

url ="https://api.open-meteo.com/v1/forecast"
print(url)

params= {
    "latitude": 27.7172,
    "longitude": 85.3240,
    "daily":[
        "temperature_2m_max",
        "temperature_2m_min",
    ],
    "timezone" : "Asia/Kathmandu"
}

https://api.open-meteo.com/v1/forecast


In [3]:
#send api get request and error handling

try:
    response = requests.get(url, params=params, timeout = 30)

    response.raise_for_status() #raise an exception for 4xx and 5xx status codes

    weather_data = response.json() #convert the response into Python objects

    print("API extraction successful!")

except requests.exceptions.Timeout:
    print("The API request timed out.")

except requests.exceptions.HTTPError as error:
    print("HTTP error:", error)

except requests.exceptions.RequestException as error:
    print("Error occurred during the API request:", error)

except ValueError as error:
    print("Error occurred while parsing the JSON response:", error)


API extraction successful!


In [4]:
#inspect the response

print("Response status code:", response.status_code)
print("Requested URL:", response.url)

Response status code: 200
Requested URL: https://api.open-meteo.com/v1/forecast?latitude=27.7172&longitude=85.324&daily=temperature_2m_max&daily=temperature_2m_min&timezone=Asia%2FKathmandu


In [5]:
print(type(weather_data)) #converted the JSON reponse body into Python dictionary

<class 'dict'>


In [6]:
print(json.dumps(weather_data, indent=4)) #pretty print the JSON response body

{
    "latitude": 27.732864,
    "longitude": 85.34831,
    "generationtime_ms": 0.09763240814208984,
    "utc_offset_seconds": 20700,
    "timezone": "Asia/Kathmandu",
    "timezone_abbreviation": "GMT+5:45",
    "elevation": 1301.0,
    "daily_units": {
        "time": "iso8601",
        "temperature_2m_max": "\u00b0C",
        "temperature_2m_min": "\u00b0C"
    },
    "daily": {
        "time": [
            "2026-11-01",
            "2026-11-02",
            "2026-11-03",
            "2026-11-04",
            "2026-11-05",
            "2026-11-06",
            "2026-11-07"
        ],
        "temperature_2m_max": [
            24.0,
            23.8,
            23.1,
            23.4,
            23.1,
            23.2,
            23.1
        ],
        "temperature_2m_min": [
            12.6,
            13.1,
            11.8,
            10.1,
            10.3,
            9.9,
            9.9
        ]
    }
}


In [7]:
#Access the daily dictionary

daily_data = weather_data["daily"]
print(type(daily_data)) 
print(daily_data.keys())

#acessing indivuduals lists

dates = daily_data["time"]
max_temps = daily_data["temperature_2m_max"]
min_temps = daily_data["temperature_2m_min"]

print("Dates:", dates)
print("Max Temperatures:", max_temps)
print("Min Temperatures:", min_temps)
                       

<class 'dict'>
dict_keys(['time', 'temperature_2m_max', 'temperature_2m_min'])
Dates: ['2026-11-01', '2026-11-02', '2026-11-03', '2026-11-04', '2026-11-05', '2026-11-06', '2026-11-07']
Max Temperatures: [24.0, 23.8, 23.1, 23.4, 23.1, 23.2, 23.1]
Min Temperatures: [12.6, 13.1, 11.8, 10.1, 10.3, 9.9, 9.9]


In [8]:
#inspecting the lengths

print("Length of Dates:", len(dates))
print("Length of Max Temperatures:", len(max_temps))
print("Length of Min Temperatures:", len(min_temps))

Length of Dates: 7
Length of Max Temperatures: 7
Length of Min Temperatures: 7


In [9]:
#Extracted the data into a structured format

weather_df = pd.DataFrame({
    "Date": dates,  
    "temperature_2m_max": max_temps,
    "temperature_2m_min": min_temps
})

weather_df

,Date,temperature_2m_max,temperature_2m_min
0,2026-11-01,24.0,12.6
1,2026-11-02,23.8,13.1
2,2026-11-03,23.1,11.8
3,2026-11-04,23.4,10.1
4,2026-11-05,23.1,10.3
5,2026-11-06,23.2,9.9
6,2026-11-07,23.1,9.9


In [10]:
#validating the data

print("Shape of the dataset:", weather_df.shape)

print("Column names:", weather_df.columns)
print("Datatypes: ", weather_df.dtypes)
print("Dataset information:", weather_df.info())

Shape of the dataset: (7, 3)
Column names: Index(['Date', 'temperature_2m_max', 'temperature_2m_min'], dtype='str')
Datatypes:  Date                      str
temperature_2m_max    float64
temperature_2m_min    float64
dtype: object
<class 'pandas.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 3 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Date                7 non-null      str    
 1   temperature_2m_max  7 non-null      float64
 2   temperature_2m_min  7 non-null      float64
dtypes: float64(2), str(1)
memory usage: 300.0 bytes
Dataset information: None


In [11]:
#convert the date into datatime format

weather_df["Date"]= pd.to_datetime(weather_df["Date"])
print(weather_df.dtypes)

Date                  datetime64[us]
temperature_2m_max           float64
temperature_2m_min           float64
dtype: object


In [12]:
#checking for missing values

print(weather_df.isnull().sum())

Date                  0
temperature_2m_max    0
temperature_2m_min    0
dtype: int64


In [13]:
#checking  the temperature consistency

invalid_records = weather_df[ 
    weather_df["temperature_2m_max"] < weather_df["temperature_2m_min"]
    ]

print("Invalid records:",invalid_records)

Invalid records: Empty DataFrame
Columns: [Date, temperature_2m_max, temperature_2m_min]
Index: []


In [14]:
#Transforming the dataset

#creating a derived column

weather_df["temperature_range"] = weather_df["temperature_2m_max"] - weather_df["temperature_2m_min"]
weather_df

                                            

,Date,temperature_2m_max,temperature_2m_min,temperature_range
0,2026-11-01,24.0,12.6,11.4
1,2026-11-02,23.8,13.1,10.7
2,2026-11-03,23.1,11.8,11.3
3,2026-11-04,23.4,10.1,13.3
4,2026-11-05,23.1,10.3,12.8
5,2026-11-06,23.2,9.9,13.3
6,2026-11-07,23.1,9.9,13.2


In [15]:
# #saving the transformed dataset to a CSV file

from  pathlib import Path

data_dir = Path.cwd().parent / "data"
output_path = data_dir / "kathmandu_weather_data.csv"

weather_df.to_csv(
    output_path,
    index=False #tells not to write index columns
)
print("Saved to:", output_path)
print("File exists:", output_path.exists())

Saved to: g:\weather-data-pipeline\data\kathmandu_weather_data.csv
File exists: True


Making Reusable Python Functions

In [16]:
def calculate_temperature_range(max_temp, min_temp):

    return max_temp - min_temp

In [17]:
result = calculate_temperature_range(30, 20)
print(result)

10


In [18]:
#applying it to pandas dataframe

weather_df["temperature_range"] = weather_df.apply(
    lambda row : calculate_temperature_range(
        row["temperature_2m_max"],
        row["temperature_2m_min"]
    ),
    axis = 1
)

Building a Reusable API Extraction Funtions

In [19]:
def extract_weather_data(latitude, longitude, timezone="auto"):

    url = "https://api.open-meteo.com/v1/forecast"
    
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "daily": [
            "temperature_2m_max",
            "temperature_2m_min",
        ],
        "timezone": timezone,
        "past_days" : 7 ,
        "forecast_days" : 3
    }

    try:
        with requests.Session() as session:
            response = session.get(url, params=params, timeout=30)
            response.raise_for_status()
            return response.json()

    except requests.exceptions.Timeout as error:
        raise RuntimeError("The API request timed out") from error
    except requests.exceptions.HTTPError as error:
        raise RuntimeError(f"HTTP error occurred: {error}") from error
    except requests.exceptions.RequestException as error:
        raise RuntimeError(f"Error occurred during the API request: {error}") from error
    except ValueError as error:
        raise RuntimeError("Error occurred while parsing the JSON response") from error

In [20]:
#testing with kathmandu

kathmandu_data = extract_weather_data(
    latitude=27.7172,
    longitude=85.3240, 
    timezone="Asia/Kathmandu")

print(kathmandu_data.keys())

pokhara_data = extract_weather_data(
    latitude=28.2096,
    longitude=83.9856,
    timezone="Asia/Kathmandu"
)

print(pokhara_data["daily"])

dict_keys(['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'daily_units', 'daily'])
{'time': ['2026-10-25', '2026-10-26', '2026-10-27', '2026-10-28', '2026-10-29', '2026-10-30', '2026-10-31', '2026-11-01', '2026-11-02', '2026-11-03'], 'temperature_2m_max': [24.1, 24.9, 25.9, 26.3, 27.1, 22.8, 22.8, 24.3, 24.5, 24.5], 'temperature_2m_min': [13.6, 12.3, 13.6, 12.1, 16.6, 16.3, 14.8, 14.5, 15.2, 13.3]}


In [21]:
london_data = extract_weather_data(
    latitude = 51.5072,
    longitude =-0.1276,
    timezone = "Europe/London" 
)
print(london_data.keys())
print(london_data["timezone"])
print(london_data["daily"]["temperature_2m_max"])

dict_keys(['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'daily_units', 'daily'])
Europe/London
[13.6, 16.8, 16.8, 15.8, 16.0, 16.2, 15.0, 12.6, 11.9, 12.8]


Creating the API validation section

In [ ]:
def validate_weather_response(data):

    #check for response structure
    if not isinstance(data, dict):
        raise ValueError("API response is not a dictionary as expected.")

    #check if the daily field exists
    if "daily" not in data:
        raise ValueError("Missing 'daily' field in API response.")

    daily_data = data["daily"]

    #check: are the 3 required fields are present

    required_fields = ["time" , "temperature_2m_max", "temperature_2m_min"]
    for field in required_fields:
        if field not in daily_data:
            raise ValueError(f"Missing required fields: {field}")

    #check for all three required fields containing python lists
    
    for field in required_fields:
            if not isinstance(daily_data[field] , list):
                raise ValueError(f"Field '{field}' is not a list as expected.")
    #check for length consistency of the lists

    lengths = [len(daily_data[field]) for field in required_fields]

    if len(set(lengths)) != 1:
        raise ValueError("Weather data arrays have unequal lengths.")

    return True
     

In [23]:
validate_weather_response(kathmandu_data)

True

In [ ]:
# #testing with invalid data
# invalid_data = {
#     "daily": {
#         "time": ["2026-10-31", "2026-11-01"],
#         "temperature_2m_max": [25.4, 26.1]
#     }
# }

# validate_weather_response(invalid_data)

# invalid_data_2 = {
#     "daily": {
#         "time": ["2026-10-31", "2026-11-01"],
#         "temperature_2m_max": [25.4],
#         "temperature_2m_min": [17.2, 18.0]
#     }
# }

# validate_weather_response(invalid_data_2)


# invalid_data_2 = {
#     "longitude": 85.3240,
#     "latitude": 27.7172
# }

# validate_weather_response(invalid_data_2)

# invalid_data_2 = {
#     "daily": {
#         "time": ["2026-10-31", "2026-11-01"],
#         "temperature_2m_max": (25.4 , 34.3),
#         "temperature_2m_min": [17.2, 18.0]
#     }
# }

# validate_weather_response(invalid_data_2)



[24.0, 23.8, 23.1, 23.4, 23.1, 23.2, 23.1]
